# Traces en vrac pour check cas limites afficliation

In [ ]:
# ========= Diagnostic des cas concernés par le fallback ==========

# Cas concernés : affiliation mandat manquante, mais groupeAbrev disponible
mask_fallback = df["affiliation_mandat_députés"].isna() & df["groupeAbrev"].notna()

fallback_cases = df.loc[
    mask_fallback,
    [
        "id_acteur",
        "nom_orateur_clean",
        "qualite_orateur",
        "groupeAbrev",
        "dateSeance_ts",
    ],
].copy()

# Reconstituer l'affiliation qui serait attribuée par le fallback
fallback_cases["affiliation_fallback"] = fallback_cases["groupeAbrev"]
fallback_cases["affiliation_fallback"] = fallback_cases["affiliation_fallback"].replace(
    recodage_affiliation
)
fallback_cases["affiliation_fallback"] = fallback_cases["affiliation_fallback"].replace(
    {"LES-REP": "LR", "UMP": "LR"}
)

# Print des infos
print("=== Cas concernés par le fallback via groupeAbrev ===")
print("Nombre d'interventions concernées :", len(fallback_cases))
print("Nombre d'id_acteur uniques :", fallback_cases["id_acteur"].nunique(dropna=True))
print(
    "Nombre d'orateurs uniques :",
    fallback_cases["nom_orateur_clean"].nunique(dropna=True),
)

print("\nListe des orateurs concernés :")
print(fallback_cases["nom_orateur_clean"].dropna().unique())

display(
    fallback_cases[
        [
            "id_acteur",
            "nom_orateur_clean",
            "groupeAbrev",
            "affiliation_fallback",
        ]
    ]
    .value_counts()
    .reset_index(name="n_interventions")
)

In [ ]:
# ========= Cas limites GOUV =============

# DANS CAS AFFILIATION DYNAMIQUE DÉPUTÉS QUI SONT GOUV

# vérification des cas sans affiliation :
print(
    "Nombre d'id_acteur uniques sans affiliation :",
    df[df["affiliation_mandat_députés"].isna()]["id_acteur"].nunique(),
)
print("\nValeur counts des id_acteur sans affiliation (top):")
print(
    df[df["affiliation_mandat_députés"].isna()][["id_acteur", "nom_orateur_clean"]]
    .value_counts()
    .head()
)

# Vérifier les cas où affiliation n'est pas nulle mais avec qualité orateur spécifique
# = membres du gouv mais qui sont députés et flaguent donc avec une affiliation députés

mask_affil_with_qualite = df["affiliation_mandat_députés"].notna() & df[
    "qualite_orateur"
].str.contains("ministre|garde des sceaux|secrétaire d'État", case=False, na=False)

print(
    "Nombre de lignes avec affiliation ET qualité gouvernementale :",
    mask_affil_with_qualite.sum(),
)
print("\nAffiliations pour ces cas :")
print(
    df.loc[mask_affil_with_qualite, "affiliation_mandat_députés"].value_counts().head()
)

print("\nExemples de ces lignes :")
print(
    df.loc[
        mask_affil_with_qualite,
        ["nom_orateur_clean", "qualite_orateur", "affiliation_mandat_députés"],
    ]
    .drop_duplicates()
    .head()
)
